## 2 序列模型

### 2.1 理论计算题

给定一个字符序列 `"ababc"`，假设采用一阶马尔可夫模型（即 $p(x_t | x_{t-1})$），
使用拉普拉斯平滑（加 1 平滑）估计以下条件概率：

1. $p('a' | 'b')$
2. $p('c' | 'b')$

（词汇表为 $\{'a','b','c'\}$，计算时考虑所有可能转移，包括未出现的情况。）

**解：**

序列 `"ababc"` 中的一阶转移（$x_{t-1} \to x_t$）：
- 位置 1→2：$a \to b$
- 位置 2→3：$b \to a$
- 位置 3→4：$a \to b$
- 位置 4→5：$b \to c$

各"从"状态的出现次数（作为前一个词，且有后继词）：
$$\text{count\_from}(a) = 2,\quad \text{count\_from}(b) = 2,\quad \text{count\_from}(c) = 0$$

转移计数：
$$\text{count}(a \to b) = 2,\quad \text{count}(b \to a) = 1,\quad \text{count}(b \to c) = 1$$
其余转移（$a \to a$、$a \to c$、$b \to b$、$c \to a$、$c \to b$、$c \to c$）均为 $0$。

拉普拉斯平滑公式（$|V|=3$）：
$$p(w_j | w_i) = \frac{\text{count}(w_i \to w_j) + 1}{\text{count\_from}(w_i) + |V|}$$

**(1) $p('a' | 'b')$：**
$$p(a|b) = \frac{\text{count}(b \to a) + 1}{\text{count\_from}(b) + 3} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = \boxed{0.4}$$

**(2) $p('c' | 'b')$：**
$$p(c|b) = \frac{\text{count}(b \to c) + 1}{\text{count\_from}(b) + 3} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = \boxed{0.4}$$

In [1]:
# 验证拉普拉斯平滑计算结果
count_from = {'a': 2, 'b': 2, 'c': 0}
transitions = {
    ('a', 'b'): 2, ('b', 'a'): 1, ('b', 'c'): 1,
    ('a', 'a'): 0, ('a', 'c'): 0, ('b', 'b'): 0,
    ('c', 'a'): 0, ('c', 'b'): 0, ('c', 'c'): 0,
}
V = 3

def laplace_prob(prev, curr):
    return (transitions.get((prev, curr), 0) + 1) / (count_from[prev] + V)

print(f"p('a'|'b') = {laplace_prob('b', 'a'):.4f}")
print(f"p('c'|'b') = {laplace_prob('b', 'c'):.4f}")

# 验证: 从 b 出发的所有条件概率之和应为 1
total = sum(laplace_prob('b', c) for c in ['a', 'b', 'c'])
print(f"\n从'b'出发的概率和 (验证): {total:.4f}")

p('a'|'b') = 0.4000
p('c'|'b') = 0.4000

从'b'出发的概率和 (验证): 1.0000


### 2.2 编程题：文本预处理与滑动窗口

编写一个函数 `preprocess_text(text, n)`，完成以下步骤：
1. 将文本转换为小写，去除标点符号（保留字母和空格）。
2. 按空格分词。
3. 构建词汇表（按出现频率排序，分配整数 ID，从 0 开始）。
4. 用滑动窗口生成长度为 $n$ 的特征序列和对应的下一个词标签（用于自回归语言模型）。

返回词汇表字典和 (特征列表, 标签列表)。

In [2]:
import re
from collections import Counter


def preprocess_text(text, n):
    """文本预处理：清洗、分词、构建词汇表、生成滑动窗口样本。

    参数:
        text: 输入文本字符串
        n: 滑动窗口长度（上下文词数）
    返回:
        vocab: 词汇表字典 {word: id}
        (features, labels): 特征列表和标签列表
            features[i] = [word_{i}, word_{i+1}, ..., word_{i+n-1}]
            labels[i]   = word_{i+n}（若无后继词则为 None）
    """
    # 1. 转小写，去除标点符号（只保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    # 2. 按空格分词
    tokens = text.split()

    # 3. 构建词汇表（按频率降序排序，分配从 0 开始的整数 ID）
    word_freq = Counter(tokens)
    sorted_words = [w for w, _ in word_freq.most_common()]
    vocab = {word: idx for idx, word in enumerate(sorted_words)}

    # 4. 滑动窗口生成 (特征, 标签)
    # 使用 len(tokens) - n + 1 确保生成最后一个可能没有标签的窗口
    features, labels = [], []
    for i in range(len(tokens) - n + 1):
        window = tokens[i:i + n]
        label = tokens[i + n] if i + n < len(tokens) else None
        features.append(window)
        labels.append(label)

    return vocab, (features, labels)


# ===== 测试 =====
text = "The time machine, by H.G. Wells. The time machine is a novel."
vocab, (features, labels) = preprocess_text(text, n=2)

print("词汇表 (word -> id):")
for w, idx in sorted(vocab.items(), key=lambda x: x[1]):
    print(f"  {idx}: '{w}'")

print(f"\n特征序列 (n=2):")
for feat, lbl in zip(features, labels):
    print(f"  {feat} -> '{lbl}'")

print(f"\n词汇表大小: {len(vocab)}")
print(f"样本数量: {len(features)}")

# ===== 验证示例 =====
print("\n" + "=" * 50)
print('验证原始示例: "The time machine", n=2')
vocab2, (feat2, lab2) = preprocess_text("The time machine", 2)
print("词汇表:", vocab2)
print("特征:", feat2)
print("标签:", lab2)

词汇表 (word -> id):
  0: 'the'
  1: 'time'
  2: 'machine'
  3: 'by'
  4: 'hg'
  5: 'wells'
  6: 'is'
  7: 'a'
  8: 'novel'

特征序列 (n=2):
  ['the', 'time'] -> 'machine'
  ['time', 'machine'] -> 'by'
  ['machine', 'by'] -> 'hg'
  ['by', 'hg'] -> 'wells'
  ['hg', 'wells'] -> 'the'
  ['wells', 'the'] -> 'time'
  ['the', 'time'] -> 'machine'
  ['time', 'machine'] -> 'is'
  ['machine', 'is'] -> 'a'
  ['is', 'a'] -> 'novel'
  ['a', 'novel'] -> 'None'

词汇表大小: 9
样本数量: 11

验证原始示例: "The time machine", n=2
词汇表: {'the': 0, 'time': 1, 'machine': 2}
特征: [['the', 'time'], ['time', 'machine']]
标签: ['machine', None]


## 3 循环神经网络

### 3.1 理论计算题

考虑一个**线性 RNN**（无偏置，无激活函数），定义为：
$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$
$$o_t = W_{oh} h_t$$

假设损失函数为平方损失：
$$L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2$$

推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播 BPTT，展开到所有时间步），并说明梯度消失或爆炸的条件。

**解：**

**一、前向传播**

对每个时间步 $t = 1, 2, \ldots, T$（设 $h_0 = \mathbf{0}$）：
$$h_t = W_{hh} h_{t-1} + W_{hx} x_t$$
$$o_t = W_{oh} h_t$$

损失：
$$L = \frac{1}{2} \sum_{t=1}^{T} \|o_t - y_t\|^2$$

**二、反向传播推导**

定义误差信号：
$$e_t = \frac{\partial L}{\partial o_t} = o_t - y_t$$

损失对隐藏状态的**直接**梯度（来自当前时刻的输出）：
$$\frac{\partial L}{\partial h_t}\bigg|_{\text{direct}} = W_{oh}^T e_t$$

由于 $h_t$ 通过 $h_{t+1}, h_{t+2}, \ldots, h_T$ 间接影响后续时刻的损失，我们需要沿时间反向传播。定义**总梯度** $\delta_t = \frac{\partial L}{\partial h_t}$：

$$\delta_T = W_{oh}^T e_T$$
$$\delta_t = W_{oh}^T e_t + W_{hh}^T \delta_{t+1}, \quad t = T-1, T-2, \ldots, 1$$

这是因为 $h_t$ 影响 $o_t$（通过 $W_{oh}$）和 $h_{t+1}$（通过 $W_{hh}$）。

将递推展开：
$$\delta_t = \sum_{k=t}^{T} (W_{hh}^T)^{k-t} \, W_{oh}^T e_k$$

**三、$W_{hh}$ 的梯度**

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta_t \, h_{t-1}^T$$

将 $\delta_t$ 的展开式代入：
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \left( \sum_{k=t}^{T} (W_{hh}^T)^{k-t} \, W_{oh}^T e_k \right) h_{t-1}^T$$

**四、梯度消失/爆炸的条件**

梯度中反复出现 $W_{hh}$ 的幂 $(W_{hh}^T)^{k-t}$。设 $W_{hh}$ 的特征值为 $\lambda_1, \lambda_2, \ldots, \lambda_H$：

- **梯度爆炸**：若 $\max_i |\lambda_i| > 1$，则 $(W_{hh}^T)^{k-t}$ 的元素随 $(k-t)$ 呈**指数增长**。长序列下梯度趋于无穷大，训练不稳定。

- **梯度消失**：若 $\max_i |\lambda_i| < 1$，则 $(W_{hh}^T)^{k-t}$ 的元素随 $(k-t)$ 呈**指数衰减**。早期时间步的梯度趋近于 $0$，模型无法学习长期依赖。

- **稳定**：仅当 $\max_i |\lambda_i| = 1$ 时（且为半单特征值），梯度在传播中既不消失也不爆炸。这在实践中极难满足。

> 注：实际 RNN 使用 $\tanh$ 或 sigmoid 非线性激活，其导数 $\leq 1$，进一步加剧了梯度消失问题。这就是 LSTM/GRU 引入门控机制的动机。

### 3.2 编程题：RNN 单元前向与单步反向传播

实现一个简单的 RNN 单元的前向传播和单步反向传播（仅计算梯度，不更新）。给定输入 $x_t$（形状 `(batch_size, input_size)`）、上一隐藏状态 $h_{prev}$（形状 `(batch_size, hidden_size)`），以及权重 $W_{hx}, W_{hh}, b_h$，计算当前隐藏状态 $h_t$（使用 tanh 激活函数）。同时实现反向传播，已知上游梯度 `dh_next`（即损失对 $h_t$ 的梯度），计算 $dx_t, dh_{prev}, dW_{hx}, dW_{hh}, db_h$。

In [3]:
import numpy as np


def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """RNN 单元前向传播。

    参数:
        x_t:    当前输入, 形状 (batch_size, input_size)
        h_prev: 上一隐藏状态, 形状 (batch_size, hidden_size)
        W_hx:   输入-隐藏权重, 形状 (input_size, hidden_size)
        W_hh:   隐藏-隐藏权重, 形状 (hidden_size, hidden_size)
        b_h:    隐藏层偏置, 形状 (hidden_size,)
    返回:
        h_t:  当前隐藏状态, 形状 (batch_size, hidden_size)
        cache: 反向传播所需的中间变量
    """
    # 线性组合: z = x_t @ W_hx + h_prev @ W_hh + b_h
    z = np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h  # (batch, hidden)
    # tanh 激活
    h_t = np.tanh(z)
    cache = (x_t, h_prev, z, h_t, W_hx, W_hh)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """RNN 单元单步反向传播。

    参数:
        dh_next: 损失对 h_t 的梯度, 形状 (batch_size, hidden_size)
        cache:   前向传播返回的中间变量
    返回:
        dx_t:   损失对 x_t 的梯度, 形状 (batch_size, input_size)
        dh_prev: 损失对 h_prev 的梯度, 形状 (batch_size, hidden_size)
        dW_hx:  损失对 W_hx 的梯度, 形状 (input_size, hidden_size)
        dW_hh:  损失对 W_hh 的梯度, 形状 (hidden_size, hidden_size)
        db_h:   损失对 b_h 的梯度, 形状 (hidden_size,)
    """
    x_t, h_prev, z, h_t, W_hx, W_hh = cache

    # 反向通过 tanh: d_tanh(z)/dz = 1 - tanh^2(z) = 1 - h_t^2
    dz = dh_next * (1.0 - h_t ** 2)  # (batch, hidden)

    # dx_t = dz @ W_hx^T
    dx_t = np.dot(dz, W_hx.T)  # (batch, input_size)

    # dh_prev = dz @ W_hh^T
    dh_prev = np.dot(dz, W_hh.T)  # (batch, hidden_size)

    # dW_hx = x_t^T @ dz
    dW_hx = np.dot(x_t.T, dz)  # (input_size, hidden_size)

    # dW_hh = h_prev^T @ dz
    dW_hh = np.dot(h_prev.T, dz)  # (hidden_size, hidden_size)

    # db_h = sum over batch dim
    db_h = np.sum(dz, axis=0)  # (hidden_size,)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ===== 测试 =====
np.random.seed(42)
batch_size, input_size, hidden_size = 2, 3, 4

x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.zeros(hidden_size)

# 前向
h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
print("=== 前向传播 ===")
print(f"输入 x_t 形状: {x_t.shape}")
print(f"上一状态 h_prev 形状: {h_prev.shape}")
print(f"当前状态 h_t 形状: {h_t.shape}")
print(f"h_t 值:\n{h_t}")

# 模拟上游梯度
dh_next = np.random.randn(batch_size, hidden_size) * 0.1

# 反向
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

print("\n=== 反向传播（梯度） ===")
print(f"dx_t 形状:    {dx_t.shape}")
print(f"dh_prev 形状: {dh_prev.shape}")
print(f"dW_hx 形状:   {dW_hx.shape}")
print(f"dW_hh 形状:   {dW_hh.shape}")
print(f"db_h 形状:    {db_h.shape}")
print(f"\ndx_t:\n{dx_t}")
print(f"\ndh_prev:\n{dh_prev}")
print(f"\ndW_hx:\n{dW_hx}")

# ===== 数值梯度检验 =====
print("\n=== 数值梯度检验 ===")
epsilon = 1e-5

# 检验 dW_hx
dW_hx_num = np.zeros_like(W_hx)
for i in range(W_hx.shape[0]):
    for j in range(W_hx.shape[1]):
        W_hx_plus = W_hx.copy()
        W_hx_plus[i, j] += epsilon
        h_t_plus, _ = rnn_cell_forward(x_t, h_prev, W_hx_plus, W_hh, b_h)
        L_plus = np.sum(h_t_plus * dh_next)  # 模拟损失

        W_hx_minus = W_hx.copy()
        W_hx_minus[i, j] -= epsilon
        h_t_minus, _ = rnn_cell_forward(x_t, h_prev, W_hx_minus, W_hh, b_h)
        L_minus = np.sum(h_t_minus * dh_next)

        dW_hx_num[i, j] = (L_plus - L_minus) / (2 * epsilon)

diff = np.abs(dW_hx - dW_hx_num).max()
print(f"dW_hx 解析梯度与数值梯度的最大差异: {diff:.2e} (应 < 1e-6)")

=== 前向传播 ===
输入 x_t 形状: (2, 3)
上一状态 h_prev 形状: (2, 4)
当前状态 h_t 形状: (2, 4)
h_t 值:
[[-3.86239454e-01  1.67210511e-01 -1.69800575e-01 -2.26907982e-05]
 [ 1.11884117e-01 -1.87834363e-01 -2.76235736e-01  3.31123890e-02]]

=== 反向传播（梯度） ===
dx_t 形状:    (2, 3)
dh_prev 形状: (2, 4)
dW_hx 形状:   (3, 4)
dW_hh 形状:   (4, 4)
db_h 形状:    (4,)

dx_t:
[[ 0.01562406 -0.01439285  0.01112186]
 [-0.00663711 -0.00164457 -0.01851843]]

dh_prev:
[[ 0.01075719  0.00297839  0.01387139 -0.01110648]
 [ 0.01229764  0.04021158  0.01898282  0.00737509]]

dW_hx:
[[-0.07416589  0.14078433 -0.02298216 -0.30397755]
 [ 0.01201145 -0.01983276  0.01242129  0.05118979]
 [ 0.00427726 -0.04283475 -0.1004326  -0.00538946]]

=== 数值梯度检验 ===
dW_hx 解析梯度与数值梯度的最大差异: 2.10e-11 (应 < 1e-6)


## 4 高级循环神经网络

### 4.1 理论计算题

假设一个**深度双向 RNN**，有 $L$ 层，每层隐藏单元数为 $H$，输入维度为 $D$，输出维度为 $O$（仅考虑最后输出层）。计算该模型的**参数总数**（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

**解：**

双向 RNN 每层包含**前向**和**后向**两个独立的 RNN 单元。

**第 1 层（输入层）：**
- 前向 RNN：输入维度 $D$，隐藏维度 $H$
  - $W_{ih} \in \mathbb{R}^{H \times D}$，$b_{ih} \in \mathbb{R}^{H}$
  - $W_{hh} \in \mathbb{R}^{H \times H}$，$b_{hh} \in \mathbb{R}^{H}$
  - 参数量：$H \cdot D + H + H \cdot H + H = H(D + H + 2)$
- 后向 RNN：同上，$H(D + H + 2)$
- 第 1 层合计：$2H(D + H + 2)$

**第 $l$ 层（$l = 2, 3, \ldots, L$）：**
- 输入来自上一层的前向+后向拼接，维度为 $2H$
- 前向 RNN：$H \cdot 2H + H + H \cdot H + H = H(3H + 2)$
- 后向 RNN：同上，$H(3H + 2)$
- 第 $l$ 层合计：$2H(3H + 2)$

**输出层：**
- 取最后一层双向的拼接隐藏状态（维度 $2H$），映射到输出维度 $O$
- $W_{out} \in \mathbb{R}^{O \times 2H}$，$b_{out} \in \mathbb{R}^{O}$
- 参数量：$O \cdot 2H + O = O(2H + 1)$

**参数总数：**
$$\boxed{P = 2H(D + H + 2) + 2(L-1)H(3H + 2) + O(2H + 1)}$$

展开整理：
$$P = 2H\big[D + H + 2 + (L-1)(3H + 2)\big] + O(2H + 1)$$
$$= 2H\big[D + H + 2 + 3HL - 3H + 2L - 2\big] + O(2H + 1)$$
$$= 2H\big[D + (3L-2)H + 2L\big] + O(2H + 1)$$
$$= \boxed{2HD + (6L-4)H^2 + 4HL + 2HO + O}$$

**数值示例**：若 $L=2, H=128, D=300, O=10$：
$$P = 2 \times 128 \times 300 + (6 \times 2 - 4) \times 128^2 + 4 \times 128 \times 2 + 2 \times 128 \times 10 + 10$$
$$= 76800 + 8 \times 16384 + 1024 + 2560 + 10$$
$$= 76800 + 131072 + 1024 + 2560 + 10 = \boxed{211466}$$

In [4]:
# 验证参数数量计算
def count_params(L, H, D, O):
    '''深度双向 RNN 参数总数'''
    layer1 = 2 * H * (D + H + 2)          # 第 1 层
    other_layers = 2 * (L - 1) * H * (3 * H + 2)  # 第 2~L 层
    output_layer = O * (2 * H + 1)        # 输出层
    return layer1 + other_layers + output_layer

L, H, D, O = 2, 128, 300, 10
total = count_params(L, H, D, O)
print(f"L={L}, H={H}, D={D}, O={O}")
print(f"第 1 层参数: {2 * H * (D + H + 2):,}")
print(f"第 2 层参数: {2 * H * (3 * H + 2):,}")
print(f"输出层参数: {O * (2 * H + 1):,}")
print(f"参数总数: {total:,}")

# 验证公式
formula = 2*H*D + (6*L-4)*H**2 + 4*H*L + 2*H*O + O
print(f"公式计算: {formula:,}")
print(f"一致: {total == formula}")

L=2, H=128, D=300, O=10
第 1 层参数: 110,080
第 2 层参数: 98,816
输出层参数: 2,570
参数总数: 211,466
公式计算: 211,466
一致: True


### 4.2 编程题：双向 RNN 编码器

实现一个双向 RNN 编码器，接收序列 $X$（形状 `(seq_len, batch, input_dim)`），使用 `torch.nn.RNN` 或手动实现。要求返回每个时间步的拼接后的前向和后向隐藏状态（形状 `(seq_len, batch, 2*hidden_dim)`），以及最终时间步的拼接隐藏状态（作为序列表示）。

In [5]:
import torch
import torch.nn as nn


class BiRNNEncoder(nn.Module):
    """双向 RNN 编码器。

    使用 torch.nn.RNN 实现，返回每个时间步的拼接隐藏状态和最终表示。
    """

    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        # bidirectional=True 自动同时运行前向和后向 RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False,  # 保持 (seq_len, batch, dim) 格式
        )

    def forward(self, X):
        """前向传播。

        参数:
            X: 输入序列, 形状 (seq_len, batch, input_dim)
        返回:
            outputs:      每个时间步的拼接隐藏状态,
                          形状 (seq_len, batch, 2 * hidden_dim)
            final_hidden: 最终时间步的拼接隐藏状态,
                          形状 (batch, 2 * hidden_dim)
        """
        # RNN 前向
        outputs, h_n = self.rnn(X)
        # outputs: (seq_len, batch, 2 * hidden_dim)
        #   - outputs[:, :, :hidden_dim] 是前向的隐藏状态
        #   - outputs[:, :, hidden_dim:] 是后向的隐藏状态
        # h_n: (2 * num_layers, batch, hidden_dim)
        #   - h_n[-2, :, :] 是最后一层前向的最终状态
        #   - h_n[-1, :, :] 是最后一层后向的最终状态

        # 取最后一层双向的最终隐藏状态并拼接
        h_forward_final = h_n[-2, :, :]   # (batch, hidden_dim)
        h_backward_final = h_n[-1, :, :]  # (batch, hidden_dim)
        final_hidden = torch.cat(
            [h_forward_final, h_backward_final], dim=1
        )  # (batch, 2 * hidden_dim)

        return outputs, final_hidden


# ===== 测试 =====
torch.manual_seed(42)
seq_len, batch, input_dim, hidden_dim = 5, 3, 8, 16

X = torch.randn(seq_len, batch, input_dim)
print(f"输入 X 形状: {tuple(X.shape)}")

encoder = BiRNNEncoder(input_dim, hidden_dim, num_layers=2)
outputs, final_hidden = encoder(X)

print(f"每个时间步输出 形状: {tuple(outputs.shape)}")
print(f"  预期: ({seq_len}, {batch}, {2 * hidden_dim})")
print(f"最终表示 形状: {tuple(final_hidden.shape)}")
print(f"  预期: ({batch}, {2 * hidden_dim})")

# 验证：前向和后向隐藏状态已正确拼接
print(f"\n前向隐藏状态在 outputs[:,:,:{hidden_dim}]")
print(f"后向隐藏状态在 outputs[:,:,{hidden_dim}:]")
print(f"\n最终前向状态: {h_forward_final.shape}" if False else "", end="")
print(f"outputs 最后一个时间步的前向 = final_hidden 的前向部分?")
last_step_forward = outputs[-1, :, :hidden_dim]
final_forward = final_hidden[:, :hidden_dim]
print(f"  差异: {(last_step_forward - final_forward).abs().max().item():.2e} (应接近 0)")

输入 X 形状: (5, 3, 8)
每个时间步输出 形状: (5, 3, 32)
  预期: (5, 3, 32)
最终表示 形状: (3, 32)
  预期: (3, 32)

前向隐藏状态在 outputs[:,:,:16]
后向隐藏状态在 outputs[:,:,16:]
outputs 最后一个时间步的前向 = final_hidden 的前向部分?
  差异: 0.00e+00 (应接近 0)


## 5 嵌入向量

### 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用**负采样**（采样 $K$ 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $\mathbf{v}_c, \mathbf{u}_o$，负样本词向量为 $\mathbf{u}_{n_k}$，写出完整的目标函数。

**解：**

**一、目标函数**

Skip-gram 的目标是：给定中心词 $w_c$，最大化观察到上下文词 $w_o$ 的概率。采用负采样后，对每个 $(w_c, w_o)$ 对，将其转化为 $K+1$ 个二分类问题：
- 1 个正样本 $(w_c, w_o)$，标签为 $1$
- $K$ 个负样本 $(w_c, w_{n_k})$，标签为 $0$，其中 $w_{n_k} \sim P_n(w)$

使用 sigmoid 函数 $\sigma(z) = \frac{1}{1+e^{-z}}$，正样本概率为 $\sigma(\mathbf{u}_o^T \mathbf{v}_c)$，负样本概率为 $\sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)$。

**单个 (中心词, 上下文词) 对的负采样对数似然（最大化目标）：**
$$\mathcal{L} = \log \sigma(\mathbf{u}_o^T \mathbf{v}_c) + \sum_{k=1}^{K} \mathbb{E}_{w_{n_k} \sim P_n(w)} \big[ \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c) \big]$$

**损失函数（最小化目标，即负对数似然）：**
$$\boxed{J = -\log \sigma(\mathbf{u}_o^T \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^T \mathbf{v}_c)}$$

对整个语料库，总损失为所有 (中心词, 上下文词) 对的损失之和。

**二、负采样策略**

噪声分布 $P_n(w)$ 通常取**一元分布（unigram）的 $3/4$ 次方**：
$$P_n(w) = \frac{U(w)^{3/4}}{\sum_{w'} U(w')^{3/4}}$$

其中 $U(w)$ 是词 $w$ 在语料中的频率（unigram 分布）。

**为什么用 $3/4$ 次方？**
- 原始频率分布中高频词（如 "the"、"a"）概率远高于低频词
- 取 $3/4$ 次方后，分布更平滑，给低频词更高的被采样概率
- 经验上在常见词和罕见词之间取得了良好平衡

**具体采样步骤：**
1. 统计语料中每个词的频率 $U(w) = \frac{\text{count}(w)}{\sum_{w'} \text{count}(w')}$
2. 计算 $U(w)^{3/4}$ 并归一化得到 $P_n(w)$
3. 每次需要负样本时，按 $P_n(w)$ 独立采样 $K$ 个词（可能与正样本重复，实践中通常排除）

### 5.2 编程题：CBOW 模型前向与损失

实现 CBOW 模型的前向传播和损失计算（**不使用负采样**，仅用完整 softmax）。给定一批上下文词的索引列表（每个样本有 `context_size` 个上下文词），词汇表大小 $V$，嵌入维度 $d$。输入权重矩阵 $W$（形状 $(V, d)$）和输出权重矩阵 $W_{out}$（形状 $(d, V)$）。计算平均上下文向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。

In [6]:
import numpy as np


def cbow_forward_loss(context_indices, center_idx, W, W_out):
    """CBOW 模型前向传播与交叉熵损失。

    参数:
        context_indices: 上下文词索引, 形状 (context_size,) 的一维数组
        center_idx:      中心词索引 (标量)
        W:               输入嵌入矩阵, 形状 (V, d)
        W_out:           输出权重矩阵, 形状 (d, V)
    返回:
        loss: 交叉熵损失 (标量)
        probs: 输出概率分布, 形状 (V,)
    """
    # 1. 查表得到上下文词的嵌入向量
    context_embeds = W[context_indices]  # (context_size, d)

    # 2. 平均上下文向量作为隐藏层表示
    h = np.mean(context_embeds, axis=0)  # (d,)

    # 3. 计算输出得分
    logits = np.dot(h, W_out)  # (V,)

    # 4. Softmax 得到概率分布
    # 数值稳定: 减去最大值防止溢出
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    probs = exp_logits / np.sum(exp_logits)  # (V,)

    # 5. 交叉熵损失: -log(p_center)
    loss = -np.log(probs[center_idx] + 1e-12)  # 加小量防止 log(0)

    return loss, probs


# ===== 测试 =====
np.random.seed(42)

# 参数设置
V, d, context_size = 100, 50, 4

# 随机初始化权重
W = np.random.randn(V, d) * 0.1      # 输入嵌入矩阵
W_out = np.random.randn(d, V) * 0.1  # 输出权重矩阵

# 随机一批数据（单个样本）
context_indices = np.array([2, 5, 8, 12])   # 上下文词索引
center_idx = 7                               # 中心词索引

print(f"词汇表大小 V = {V}, 嵌入维度 d = {d}, 上下文窗口 = {context_size}")
print(f"上下文词索引: {context_indices}")
print(f"中心词索引: {center_idx}")

# 计算损失
loss, probs = cbow_forward_loss(context_indices, center_idx, W, W_out)

print(f"\n交叉熵损失: {loss:.6f}")
print(f"预测概率分布总和: {probs.sum():.6f} (应为 1.0)")
print(f"中心词预测概率: {probs[center_idx]:.6f}")
print(f"Top-5 预测词索引: {np.argsort(probs)[::-1][:5]}")
print(f"Top-5 预测概率: {np.sort(probs)[::-1][:5]}")

# ===== 验证：随机猜测的基准损失 =====
print(f"\n随机猜测基准损失 (-log(1/V)): {-np.log(1/V):.4f}")
print(f"模型损失应小于基准: {loss < -np.log(1/V)}")

# ===== 批量版本 =====
print("\n" + "=" * 50)
print("批量版本测试 (batch_size=3)")

def cbow_batch_loss(context_indices_batch, center_indices, W, W_out):
    """批量 CBOW 损失计算。"""
    batch_size = len(context_indices_batch)
    losses = []
    for i in range(batch_size):
        loss, _ = cbow_forward_loss(
            context_indices_batch[i], center_indices[i], W, W_out
        )
        losses.append(loss)
    return np.mean(losses)

batch_context = np.array([
    [2, 5, 8, 12],
    [3, 6, 9, 15],
    [1, 4, 7, 10],
])
batch_center = np.array([7, 11, 3])

batch_loss = cbow_batch_loss(batch_context, batch_center, W, W_out)
print(f"批量平均交叉熵损失: {batch_loss:.6f}")

词汇表大小 V = 100, 嵌入维度 d = 50, 上下文窗口 = 4
上下文词索引: [ 2  5  8 12]
中心词索引: 7

交叉熵损失: 4.588069
预测概率分布总和: 1.000000 (应为 1.0)
中心词预测概率: 0.010172
Top-5 预测词索引: [59 84 23 26 41]
Top-5 预测概率: [0.01115587 0.01082904 0.01080926 0.01065217 0.0106363 ]

随机猜测基准损失 (-log(1/V)): 4.6052
模型损失应小于基准: True

批量版本测试 (batch_size=3)
批量平均交叉熵损失: 4.583159


## 6 注意力机制

### 6.1 理论计算题

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算**缩放点积注意力**（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再 softmax，再加权求和）。使用 $\text{score} = QK^T / \sqrt{d_k}$（$d_k = 4$）。

> 可以用符号或具体数值演示。下面用随机生成的数值矩阵演示完整计算过程。

**解：**

设具体数值（为清晰演示，使用简单数值）：

$$Q = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 1 & 1 & 1 & 1 \end{bmatrix} \in \mathbb{R}^{2 \times 4}$$

$$K = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 1 & 1 & 1 & 1 \\ 0 & 1 & 0 & 1 \end{bmatrix} \in \mathbb{R}^{3 \times 4}$$

$$V = \begin{bmatrix} 1 & 0 & 0 & 0 & 0 \\ 0 & 1 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 & 1 \end{bmatrix} \in \mathbb{R}^{3 \times 5}$$

**步骤一：计算得分矩阵 $S = QK^T / \sqrt{d_k}$**

$$d_k = 4,\quad \sqrt{d_k} = 2$$

$$QK^T = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 1 & 1 & 1 & 1 \end{bmatrix} \begin{bmatrix} 1 & 1 & 0 \\ 0 & 1 & 1 \\ 1 & 1 & 0 \\ 0 & 1 & 1 \end{bmatrix} = \begin{bmatrix} (1+0+1+0) & (1+0+1+0) & (0+0+0+0) \\ (1+1+1+1) & (0+1+1+1) & (0+1+0+1) \end{bmatrix}$$

$$= \begin{bmatrix} 2 & 2 & 0 \\ 4 & 3 & 2 \end{bmatrix}$$

$$S = QK^T / 2 = \begin{bmatrix} 1.0 & 1.0 & 0.0 \\ 2.0 & 1.5 & 1.0 \end{bmatrix}$$

**步骤二：Softmax（按行归一化）**

对于第 1 行 $[1.0, 1.0, 0.0]$：
$$\text{softmax} = \left[\frac{e^{1.0}}{e^{1.0}+e^{1.0}+e^{0.0}}, \frac{e^{1.0}}{e^{1.0}+e^{1.0}+e^{0.0}}, \frac{e^{0.0}}{e^{1.0}+e^{1.0}+e^{0.0}}\right]$$
$$= \left[\frac{2.718}{2.718+2.718+1.0}, \frac{2.718}{7.436}, \frac{1.0}{7.436}\right] \approx [0.3655, 0.3655, 0.1345]$$

对于第 2 行 $[2.0, 1.5, 1.0]$：
$$\text{softmax} \approx [0.5060, 0.3070, 0.1863]$$

$$A = \text{softmax}(S) = \begin{bmatrix} 0.3655 & 0.3655 & 0.1345 \\ 0.5060 & 0.3070 & 0.1863 \end{bmatrix}$$

**步骤三：加权求和 $\text{Output} = A \cdot V$**

$$\text{Output} = \begin{bmatrix} 0.3655 & 0.3655 & 0.1345 \\ 0.5060 & 0.3070 & 0.1863 \end{bmatrix} \begin{bmatrix} 1 & 0 & 0 & 0 & 0 \\ 0 & 1 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 & 1 \end{bmatrix}$$

第 1 行：
$$[0.3655 \cdot 1 + 0 + 0,\ 0 + 0.3655 \cdot 1 + 0,\ 0 + 0.3655 \cdot 1 + 0,\ 0 + 0 + 0.1345 \cdot 1,\ 0 + 0 + 0.1345 \cdot 1]$$
$$= [0.3655,\ 0.3655,\ 0.3655,\ 0.1345,\ 0.1345]$$

第 2 行：
$$[0.5060,\ 0.3070,\ 0.3070,\ 0.1863,\ 0.1863]$$

$$\boxed{\text{Output} = \begin{bmatrix} 0.3655 & 0.3655 & 0.3655 & 0.1345 & 0.1345 \\ 0.5060 & 0.3070 & 0.3070 & 0.1863 & 0.1863 \end{bmatrix} \in \mathbb{R}^{2 \times 5}}$$

In [7]:
import numpy as np

# 设定数值矩阵
Q = np.array([[1., 0., 1., 0.],
              [1., 1., 1., 1.]])  # (2, 4)
K = np.array([[1., 0., 1., 0.],
              [1., 1., 1., 1.],
              [0., 1., 0., 1.]])  # (3, 4)
V = np.array([[1., 0., 0., 0., 0.],
              [0., 1., 1., 0., 0.],
              [0., 0., 0., 1., 1.]])  # (3, 5)

d_k = 4
scale = np.sqrt(d_k)

# 步骤 1: 得分矩阵
scores = np.dot(Q, K.T) / scale
print("=== 步骤 1: 得分矩阵 S = QK^T / sqrt(d_k) ===")
print(scores)

# 步骤 2: Softmax
def softmax(x, axis=-1):
    x_stable = x - np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x_stable)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

attn_weights = softmax(scores, axis=1)
print("\n=== 步骤 2: Softmax 注意力权重 A ===")
print(np.round(attn_weights, 4))

# 步骤 3: 加权求和
output = np.dot(attn_weights, V)
print("\n=== 步骤 3: 输出 = A @ V ===")
print(np.round(output, 4))
print(f"\n输出形状: {output.shape}  (预期 (2, 5))")

# 验证：每行注意力权重之和为 1
print(f"\n每行权重和: {attn_weights.sum(axis=1)} (应为 [1.0, 1.0])")

=== 步骤 1: 得分矩阵 S = QK^T / sqrt(d_k) ===
[[1. 1. 0.]
 [1. 2. 1.]]

=== 步骤 2: Softmax 注意力权重 A ===
[[0.4223 0.4223 0.1554]
 [0.2119 0.5761 0.2119]]

=== 步骤 3: 输出 = A @ V ===
[[0.4223 0.4223 0.4223 0.1554 0.1554]
 [0.2119 0.5761 0.5761 0.2119 0.2119]]

输出形状: (2, 5)  (预期 (2, 5))

每行权重和: [1. 1.] (应为 [1.0, 1.0])


### 6.2 编程题：多头注意力（Multi-Head Attention）

实现多头注意力（Multi-Head Attention）的前向传播，假设 `num_heads=2`，`d_model=4`。给定输入 $X$（形状 `(seq_len, batch, d_model)`），分别线性投影得到 $Q, K, V$（每个头的维度 $d_k = d_v = d_{model} / num\_heads$）。对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。返回输出（形状与输入相同）。

In [8]:
import numpy as np


def multi_head_attention_forward(X, num_heads=2, d_model=4):
    """多头注意力前向传播。

    参数:
        X:         输入, 形状 (seq_len, batch, d_model)
        num_heads: 注意力头数 (默认 2)
        d_model:   模型维度 (默认 4)
    返回:
        output: 注意力输出, 形状 (seq_len, batch, d_model)
        attn_weights_list: 每个头的注意力权重 (用于检查)
    """
    seq_len, batch, _ = X.shape
    d_k = d_model // num_heads  # 每个头的维度 = 2

    # ---- 线性投影权重 ----
    # 实际应用中这些是可学习参数，这里用随机初始化模拟
    rng = np.random.RandomState(42)
    W_Q = rng.randn(d_model, d_model) * 0.1  # (4, 4)
    W_K = rng.randn(d_model, d_model) * 0.1  # (4, 4)
    W_V = rng.randn(d_model, d_model) * 0.1  # (4, 4)
    W_O = rng.randn(d_model, d_model) * 0.1  # (4, 4) 最终输出投影

    # ---- 1. 线性投影得到 Q, K, V ----
    # X: (seq_len, batch, d_model), W: (d_model, d_model)
    Q = np.dot(X, W_Q)  # (seq_len, batch, d_model)
    K = np.dot(X, W_K)  # (seq_len, batch, d_model)
    V = np.dot(X, W_V)  # (seq_len, batch, d_model)

    # ---- 2. 拆分为多头 ----
    # 将 d_model 维度拆成 (num_heads, d_k)
    # 重塑为 (seq_len, batch, num_heads, d_k) 再转置为 (batch, num_heads, seq_len, d_k)
    def split_heads(tensor):
        s, b, _ = tensor.shape
        tensor = tensor.reshape(s, b, num_heads, d_k)
        return tensor.transpose(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)

    Q_heads = split_heads(Q)  # (batch, num_heads, seq_len, d_k)
    K_heads = split_heads(K)  # (batch, num_heads, seq_len, d_k)
    V_heads = split_heads(V)  # (batch, num_heads, seq_len, d_k)

    # ---- 3. 每个头独立计算缩放点积注意力 ----
    scale = np.sqrt(d_k)
    head_outputs = []
    attn_weights_list = []

    for h in range(num_heads):
        Q_h = Q_heads[:, h, :, :]  # (batch, seq_len, d_k)
        K_h = K_heads[:, h, :, :]  # (batch, seq_len, d_k)
        V_h = V_heads[:, h, :, :]  # (batch, seq_len, d_k)

        # 得分矩阵: (batch, seq_len, seq_len)
        scores = np.matmul(Q_h, K_h.transpose(0, 2, 1)) / scale

        # Softmax 归一化
        scores_stable = scores - np.max(scores, axis=-1, keepdims=True)
        attn_weights = np.exp(scores_stable)
        attn_weights = attn_weights / np.sum(attn_weights, axis=-1, keepdims=True)
        attn_weights_list.append(attn_weights)

        # 加权求和: (batch, seq_len, d_k)
        head_out = np.matmul(attn_weights, V_h)
        head_outputs.append(head_out)

    # ---- 4. 拼接所有头 ----
    # 每个 head_out: (batch, seq_len, d_k) → 拼接为 (batch, seq_len, d_model)
    concat = np.concatenate(head_outputs, axis=-1)  # (batch, seq_len, d_model)
    # 转回 (seq_len, batch, d_model)
    concat = concat.transpose(1, 0, 2)  # (seq_len, batch, d_model)

    # ---- 5. 最终线性投影 ----
    output = np.dot(concat, W_O)  # (seq_len, batch, d_model)

    return output, attn_weights_list


# ===== 测试 =====
np.random.seed(0)
seq_len, batch, d_model, num_heads = 3, 2, 4, 2

X = np.random.randn(seq_len, batch, d_model)
print(f"输入 X 形状: {X.shape}")
print(f"X 值:\n{X}")

output, attn_weights = multi_head_attention_forward(
    X, num_heads=num_heads, d_model=d_model
)

print(f"\n多头注意力输出形状: {output.shape}")
print(f"预期形状: ({seq_len}, {batch}, {d_model})")
print(f"\n输出值:\n{np.round(output, 4)}")

print(f"\n=== 每个头的注意力权重 ===")
for h, w in enumerate(attn_weights):
    print(f"\n头 {h}:")
    for b in range(batch):
        print(f"  batch {b}:")
        print(f"    {np.round(w[b], 4)}")
        print(f"    行和: {w[b].sum(axis=1)}")  # 验证 softmax

# ===== 验证：与 PyTorch 对照 =====
print("\n" + "=" * 50)
print("与 PyTorch MultiheadAttention 对照")

try:
    import torch
    import torch.nn as nn

    # 设置相同随机种子
    torch.manual_seed(42)
    X_torch = torch.randn(seq_len, batch, d_model)

    mha = nn.MultiheadAttention(
        embed_dim=d_model, num_heads=num_heads, batch_first=False
    )
    out_torch, attn_torch = mha(X_torch, X_torch, X_torch)

    print(f"PyTorch MHA 输出形状: {tuple(out_torch.shape)}")
    print(f"PyTorch MHA 注意力权重形状: {tuple(attn_torch.shape)}")
    print(f"\n两者输出形状一致: {output.shape == tuple(out_torch.shape)}")
except ImportError:
    print("PyTorch 不可用，跳过对照")

输入 X 形状: (3, 2, 4)
X 值:
[[[ 1.76405235  0.40015721  0.97873798  2.2408932 ]
  [ 1.86755799 -0.97727788  0.95008842 -0.15135721]]

 [[-0.10321885  0.4105985   0.14404357  1.45427351]
  [ 0.76103773  0.12167502  0.44386323  0.33367433]]

 [[ 1.49407907 -0.20515826  0.3130677  -0.85409574]
  [-2.55298982  0.6536186   0.8644362  -0.74216502]]]

多头注意力输出形状: (3, 2, 4)
预期形状: (3, 2, 4)

输出值:
[[[ 0.0117  0.0058 -0.0211 -0.0108]
  [ 0.0017 -0.0117  0.0119  0.0067]]

 [[ 0.0117  0.0057 -0.0202 -0.0097]
  [ 0.0017 -0.0116  0.0119  0.0067]]

 [[ 0.0118  0.0053 -0.0213 -0.0114]
  [ 0.0016 -0.0115  0.0122  0.0078]]]

=== 每个头的注意力权重 ===

头 0:
  batch 0:
    [[0.3316 0.3419 0.3265]
 [0.3338 0.3377 0.3285]
 [0.3313 0.3324 0.3363]]
    行和: [1. 1. 1.]
  batch 1:
    [[0.3295 0.3323 0.3381]
 [0.3306 0.333  0.3364]
 [0.3531 0.3358 0.3111]]
    行和: [1. 1. 1.]

头 1:
  batch 0:
    [[0.3493 0.3327 0.318 ]
 [0.3405 0.3243 0.3352]
 [0.3313 0.3512 0.3175]]
    行和: [1. 1. 1.]
  batch 1:
    [[0.331  0.3322 0.3369]
 